# Amazon Sales Exploratory Data Analysis (EDA)

Этот ноутбук содержит полноценный разведочный анализ данных (EDA) датасета продаж Amazon. В нём реализованы лучшие практики очистки данных и визуализации, необходимые для работы Data Analyst / ML Engineer.

## 1. Загрузка библиотек и данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка отображения графиков
%matplotlib inline
sns.set_theme(style="whitegrid")

# Загрузка данных (файл лежит в той же папке)
df = pd.read_csv('amazon.csv')
print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

## 2. Очистка данных (Data Cleaning)

Главная проблема исходного датасета — числа, записанные в виде строк со спецсимволами (рупии `₹`, запятые `,`, проценты `%`). Мы должны очистить их и привести к числовым типам (`float` / `int`).

In [ ]:
df_cleaned = df.copy()

def clean_currency(x):
    if isinstance(x, str):
        x = x.replace('₹', '').replace(',', '').strip()
    return pd.to_numeric(x, errors='coerce')

def clean_percentage(x):
    if isinstance(x, str):
        x = x.replace('%', '').strip()
    return pd.to_numeric(x, errors='coerce')

def clean_numeric_string(x):
    if isinstance(x, str):
        x = x.replace(',', '').strip()
    return pd.to_numeric(x, errors='coerce')

# 1. Очистка цен
df_cleaned['discounted_price'] = df_cleaned['discounted_price'].apply(clean_currency)
df_cleaned['actual_price'] = df_cleaned['actual_price'].apply(clean_currency)

# 2. Очистка процентов скидки
df_cleaned['discount_percentage'] = df_cleaned['discount_percentage'].apply(clean_percentage)

# 3. Очистка счетчика рейтингов
df_cleaned['rating_count'] = df_cleaned['rating_count'].apply(clean_numeric_string)

# 4. Очистка рейтинга (в данных есть мусорные значения типа '|')
df_cleaned['rating'] = pd.to_numeric(df_cleaned['rating'], errors='coerce')

print("Типы данных после очистки:")
print(df_cleaned[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']].dtypes)

print("\nКоличество пропусков (NaN) после конвертации:")
print(df_cleaned.isnull().sum()[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']])

## 3. Разведочный анализ данных (EDA)

Теперь мы можем строить осмысленные графики.

### 3.1 Распределение цен и скидок (Univariate Analysis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_cleaned['discounted_price'], bins=50, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Распределение цены со скидкой (Log scale)')
axes[0].set_yscale('log') # Логарифмическая шкала, так как есть очень дорогие товары

sns.histplot(df_cleaned['discount_percentage'], bins=30, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Распределение размера скидки (%)')
axes[1].set_xlabel('Скидка (%)')

sns.histplot(df_cleaned['rating'].dropna(), bins=20, kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('Распределение рейтингов')
axes[2].set_xlabel('Рейтинг (0-5)')

plt.tight_layout()
plt.show()

### 3.2 Взаимосвязь между переменными (Bivariate Analysis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df_cleaned, x='discount_percentage', y='rating', alpha=0.5, ax=axes[0], color='purple')
axes[0].set_title('Влияет ли большая скидка на рейтинг?')
axes[0].set_xlabel('Скидка (%)')
axes[0].set_ylabel('Рейтинг')

sns.scatterplot(data=df_cleaned, x='discounted_price', y='rating', alpha=0.5, ax=axes[1])
axes[1].set_title('Цена товара vs Рейтинг')
axes[1].set_xlabel('Цена (₹)')
axes[1].set_ylabel('Рейтинг')
axes[1].set_xscale('log') # Логарифмическая шкала для цены

plt.tight_layout()
plt.show()

### 3.3 Анализ категорий (Categorical Analysis)
В столбце `category` хранится полное дерево категорий (разделитель `|`). Давайте выделим главную (первую) категорию и посмотрим статистику по ней.

In [ ]:
# Извлекаем главную категорию
df_cleaned['main_category'] = df_cleaned['category'].apply(lambda x: str(x).split('|')[0] if pd.notnull(x) else 'Unknown')

# Считаем популярность категорий
category_counts = df_cleaned['main_category'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(y=category_counts.index, x=category_counts.values, palette='viridis')
plt.title('Топ 10 главных категорий товаров')
plt.xlabel('Количество товаров')
plt.ylabel('Главная категория')
plt.show()

In [ ]:
# Средний рейтинг по главным категориям
cat_stats = df_cleaned.groupby('main_category').agg({'rating': 'mean', 'discount_percentage': 'mean', 'product_id': 'count'}).reset_index()
cat_stats = cat_stats[cat_stats['product_id'] > 10] # Отсеиваем слишком редкие категории для честной статистики
cat_stats = cat_stats.sort_values(by='rating', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=cat_stats.head(10), y='main_category', x='rating', palette='magma')
plt.title('Топ категорий по среднему рейтингу (минимум 10 товаров)')
plt.xlabel('Средний рейтинг')
plt.ylabel('Категория')
plt.xlim(3.5, 4.5)
plt.show()

## 4. Корреляционная матрица

In [ ]:
plt.figure(figsize=(8, 6))
numeric_cols = ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
corr_matrix = df_cleaned[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Матрица корреляций Пирсона')
plt.show()